## PROBLEM SET 2
**Start date:03/03/2026**

Carlos Álvarez and Jordi Torres

In [1]:
##Add here folder with exports

#export_dir ""

#Note: change this manually to the folder where all the tables are printed in. 

In [2]:

using LinearAlgebra
using Statistics
using Random
using Printf

#-------------------------------------------------------------------------------
#0. Set base parameters
#-------------------------------------------------------------------------------
struct Params
    b     :: Float64    #baseline v of leisure
    zbar  :: Float64    #mean prod
    γ     :: Float64    #baseline Nash-b weiggt
    r     :: Float64    #monthly interest rate
    s     :: Float64    #monthly separation rate
    α     :: Float64    #matching function elasticity
    μ     :: Float64    #matching function constant 
    ρ     :: Float64    #AR(1) param (persistence)
    σ_z   :: Float64    #std dev of productivity process
end

function calibrate()
    b     = 0.40
    zbar  = 1.00
    α     = 0.72
    γ     = 0.72
    r     = (1.012)^(1/3) - 1      # quarterly → monthly
    s     = 0.034                   # monthly (Shimer Sec. I.D)
    μ     = 0.45                    # targets f(1)=0.45/month
    κ_q   = 0.004;  σ_q = 0.0165   # OU parameters (quarterly)
    κ_m   = κ_q / 3.0              # monthly mean reversion
    σ_m   = σ_q / sqrt(3.0)        # monthly volatility
    ρ     = exp(-κ_m)
    σ_ε   = σ_m * sqrt((1.0 - exp(-2κ_m)) / (2κ_m))
    σ_z   = σ_ε / sqrt(1.0 - ρ^2)
    return Params(b, zbar, γ, r, s, α, μ, ρ, σ_z)
end


calibrate (generic function with 1 method)

In [3]:

#-------------------------------------------------------------------------------
#1.  MODEL SPECIFICATION
#-------------------------------------------------------------------------------

#Key parameters ----------------------------------------------------------------
struct ModelSpec
    name  :: Symbol
    b_val :: Float64   # fixed leisure for :baseline, :hm, :hall-->fixed!
    ζ     :: Float64   #only for :proc_leisure:  b(z) = ζ·z -->exercise ()
    phi   :: Float64   #only for :proc_gamma:    γ(z) = γ̄ + φ·(z − z̄)-->exercise ()
end

# General functions ------------------------------------------------------------
# Note: idea is to be as general as possible in the solver and introduce here the different variants of the model to introduce there. 

#baseline
baseline_spec(par::Params) =
    ModelSpec(:baseline, par.b, 0.0, 0.0)

#Hagedorn and Manovskii (2008)
function hm_spec(par::Params, z_min::Float64)
    # Set b as high as possible while ensuring z_i − b > 0 at all grid points
    b_hm = 0.95 * z_min
    @printf("  [HM] b_hm = %.4f (z_min = %.4f, surplus margin = %.4f)\n",
            b_hm, z_min, z_min - b_hm)
    ModelSpec(:hm, b_hm, 0.0, 0.0)
end

#Hall (2005)
hall_spec(par::Params) =
    ModelSpec(:hall, par.b, 0.0, 0.0)


#Chodorow-Reich and Karabarbounis (2016) 
function proc_leisure_spec(par::Params, z_bar::Float64=1.0)
    ζ = 0.955
    @printf("  [ProcLeisure] ζ = %.3f → b(z̄) = %.4f\n", ζ, ζ*z_bar)
    ModelSpec(:proc_leisure, par.b, ζ, 0.0)
end

proc_gamma_spec(par::Params, phi::Float64=0.1) =
    ModelSpec(:proc_gamma, par.b, 0.0, phi)

# State-dependent helpers ------------------------------------------------

get_b(spec::ModelSpec, z::Float64) =
    spec.name == :proc_leisure ? spec.ζ * z : spec.b_val

function get_gamma(spec::ModelSpec, γ̄::Float64, z::Float64, z_bar::Float64)
    spec.name == :proc_gamma ?
        clamp(γ̄ + spec.phi * (z - z_bar), 1e-4, 1.0 - 1e-4) : γ̄
end

function calibrate_c(par::Params, spec::ModelSpec)
    β    = 1.0 / (1.0 + par.r)    # discount factor
    z̄    = par.zbar
    b̄    = get_b(spec, z̄)
    γ̄    = get_gamma(spec, par.γ, z̄, z̄)
    denom = (1.0/par.μ) * (1.0 - β*(1.0-par.s)) + β*γ̄
    c = (1.0 - γ̄) * (z̄ - b̄) / denom
    return c
end

function compute_wbar(par::Params, c::Float64)
    β = 1.0 / (1.0 + par.r)
    return (1.0 - par.γ)*par.b + par.γ*par.zbar + β*par.γ*c
end

#Dictionary of the names, used to print results 

spec_title(spec::ModelSpec) = Dict(
    :baseline     => "Baseline (Shimer 2005)",
    :hm           => "High Leisure (Hagedorn--Manovskii 2008)",
    :hall         => "Constant Wage (Hall 2005)",
    :proc_leisure => "Procyclical Leisure (Chodorow-Reich \\& Karabarbounis 2016)",
    :proc_gamma   => "Procyclical Bargaining Weight",
)[spec.name]


spec_title (generic function with 1 method)

In [4]:

#-------------------------------------------------------------------------------
#2. Rouwenhorst, discretization 
#-------------------------------------------------------------------------------
function rouwenhorst(N::Int, ρ::Float64, σ_z::Float64)
    ψ      = σ_z * sqrt(N - 1)
    y_grid = collect(range(-ψ, ψ, length=N))
    p_rw   = (1.0 + ρ) / 2.0          # Rouwenhorst parameter (not productivity)
    Π      = [p_rw 1-p_rw; 1-p_rw p_rw]
    for n in 3:N
        Π_old = copy(Π)
        z0    = zeros(n-1)
        A = [Π_old z0;  z0'  0.0]
        B = [z0 Π_old;  0.0  z0']
        C = [z0'  0.0;  Π_old z0]
        D = [0.0  z0';  z0 Π_old]
        Π_new = p_rw*A + (1-p_rw)*B + (1-p_rw)*C + p_rw*D
        Π_new[2:end-1, :] ./= 2.0
        Π = Π_new
    end
    π_stat = [Float64(binomial(N-1, i-1)) * 0.5^(N-1) for i in 1:N]
    π_stat ./= sum(π_stat)
    return y_grid, Π, π_stat
end

get_z_grid(y_grid, b, zbar) = b .+ exp.(y_grid) .* (zbar - b)

##J: I think there is a function that does this directly, as we studied with Alexandre . mc = rouwenhorst(N, ρ, σ) To do: update this properly to simplify a bit the lines.

get_z_grid (generic function with 1 method)

In [5]:
#-------------------------------------------------------------------------------
#3. Solve Equilibrium.  
#-------------------------------------------------------------------------------

function solve_equilibrium(par::Params, spec::ModelSpec,
                           z_grid::Vector{Float64}, Π::Matrix{Float64};
                           tol=1e-12, maxiter=20_000, ω=0.5)
    N    = length(z_grid)
    β    = 1.0 / (1.0 + par.r)    #discount factor
    z̄    = par.zbar
    c    = calibrate_c(par, spec)

    # ------------------------------------------------------------------
    if spec.name == :hall
        w̄   = compute_wbar(par, c)
        rhs = (z_grid .- w̄) ./ c
        #Direct linear solve
        A = I(N) - β*(1.0 - par.s)*Π
        x = A \ rhs
        x = max.(x, 1e-10)
        θ = (par.μ .* x).^(1.0/par.α)
        return θ, c

    # ------------------------------------------------------------------
    elseif spec.name == :proc_gamma
        γ_grid = [get_gamma(spec, par.γ, z, z̄) for z in z_grid]
        b_vals = [get_b(spec, z)                 for z in z_grid]
        rhs    = (z_grid .- b_vals) ./ c

        x = rhs ./ (1.0 - β*(1.0-par.s))    # initial guess

        for _ in 1:maxiter
            θ    = [(par.μ * (1.0-γ_grid[i]) * x[i])^(1.0/par.α) for i in 1:N]
            g    = [γ_grid[i]*θ[i] / (1.0-γ_grid[i])              for i in 1:N]
            xnew = rhs + β*(1.0-par.s)*(Π*x) - β*(Π*g)
            xnew = ω.*xnew + (1.0-ω).*x
            maximum(abs.(xnew .- x)) < tol && (x = xnew; break)
            x = xnew
        end
        θ = [(par.μ*(1.0-γ_grid[i])*x[i])^(1.0/par.α) for i in 1:N]
        return θ, c

    # ------------------------------------------------------------------
    else  # :baseline, :hm, :proc_leisure —> identical structure!
        b_vals = [get_b(spec, z) for z in z_grid]
        rhs    = (1.0 - par.γ) .* (z_grid .- b_vals) ./ c

        x = rhs ./ (1.0 - β*(1.0-par.s))    #initial guess

        for _ in 1:maxiter
            θ    = (par.μ .* x).^(1.0/par.α)
            xnew = rhs + β*(1.0-par.s)*(Π*x) - β*par.γ*(Π*θ)
            xnew = ω.*xnew + (1.0-ω).*x
            maximum(abs.(xnew .- x)) < tol && (x = xnew; break)
            x = xnew
        end
        return (par.μ .* x).^(1.0/par.α), c
    end
end



solve_equilibrium (generic function with 1 method)

In [6]:
#-------------------------------------------------------------------------------
#4. HP filter.   
#-------------------------------------------------------------------------------
function hp_filter(y::Vector{Float64}, λ::Float64)
    T = length(y)
    D = zeros(T-2, T)
    for i in 1:T-2
        D[i,i]=1.; D[i,i+1]=-2.; D[i,i+2]=1.
    end
    τ = (I(T) + λ*(D'*D)) \ y
    return τ, y .- τ
end

#To do, substitute with function that does this directly. Same as above, there is a much simpler function to do it. 

hp_filter (generic function with 1 method)

In [7]:
#-------------------------------------------------------------------------------
#5. Simulation of the economies.   
#-------------------------------------------------------------------------------

function simulate_spec(par::Params, z_grid::Vector{Float64},
                       θ_grid::Vector{Float64}, Π::Matrix{Float64};
                       n_months_data::Int = 636,
                       n_months_burn::Int = 3_000,
                       n_sims::Int        = 10_000,
                       seed::Int          = 1234)

    N       = length(z_grid)
    T_total = n_months_burn + n_months_data
    T_q     = n_months_data ÷ 3
    Random.seed!(seed)

    Π_cum  = cumsum(Π, dims=2)
    f_grid = par.μ .* max.(θ_grid, 1e-10).^(1.0 - par.α)

    std_mat  = zeros(n_sims, 5)
    auto_mat = zeros(n_sims, 5)
    corr_arr = zeros(n_sims, 5, 5)

    u_m = Vector{Float64}(undef, n_months_data)
    v_m = Vector{Float64}(undef, n_months_data)
    f_m = Vector{Float64}(undef, n_months_data)
    z_m = Vector{Float64}(undef, n_months_data)

    for sim in 1:n_sims
        state = div(N, 2) + 1
        u     = par.s / (par.s + f_grid[state])
        t_rec = 0

        for t in 1:T_total
            f_t = f_grid[state]
            if t > n_months_burn
                t_rec += 1
                u_m[t_rec] = u
                v_m[t_rec] = θ_grid[state] * u
                f_m[t_rec] = f_t
                z_m[t_rec] = z_grid[state]
            end
            u     = par.s*(1.0-u) + (1.0-f_t)*u
            draw  = rand()
            state = clamp(searchsortedfirst(view(Π_cum, state, :), draw), 1, N)
        end

        #Aggregate to quarterly
        u_q  = [mean(view(u_m, 3*(q-1)+1:3*q)) for q in 1:T_q]
        v_q  = [mean(view(v_m, 3*(q-1)+1:3*q)) for q in 1:T_q]
        f_q  = [mean(view(f_m, 3*(q-1)+1:3*q)) for q in 1:T_q]
        z_q  = [mean(view(z_m, 3*(q-1)+1:3*q)) for q in 1:T_q]
        vu_q = v_q ./ u_q

        #HP filter each log series
        series = [log.(u_q), log.(v_q), log.(vu_q), log.(f_q), log.(z_q)]
        cycles = [hp_filter(s, 1e5)[2] for s in series]

        #Compute relevant statistics.
        for k in 1:5
            std_mat[sim,k]  = std(cycles[k])
            auto_mat[sim,k] = cor(cycles[k][1:end-1], cycles[k][2:end])
        end
        corr_arr[sim,:,:] = cor(hcat(cycles...))
    end

    return std_mat, auto_mat, corr_arr
end

simulate_spec (generic function with 1 method)

In [8]:
#-------------------------------------------------------------------------------
#6. Moments helper
#-------------------------------------------------------------------------------
#To do: add intuition behind this. 

function compute_moments(std_mat, auto_mat, corr_arr)
    m_std   = vec(mean(std_mat,  dims=1))
    bs_std  = vec(std(std_mat,   dims=1))
    m_auto  = vec(mean(auto_mat, dims=1))
    bs_auto = vec(std(auto_mat,  dims=1))
    m_corr  = dropdims(mean(corr_arr, dims=1), dims=1)
    bs_corr = dropdims(std(corr_arr,  dims=1), dims=1)
    return m_std, bs_std, m_auto, bs_auto, m_corr, bs_corr
end

compute_moments (generic function with 1 method)

In [9]:
#-------------------------------------------------------------------------------
#7. To show in console. 
#-------------------------------------------------------------------------------
#To do: add intuition behind this. 

function display_table(spec::ModelSpec, m_std, bs_std, m_auto, bs_auto,
                       m_corr, bs_corr)
    labels = ["u", "v", "v/u", "f", "z"]
    K = 5
    println("\n" * "="^72)
    println(spec_title(spec))
    println("="^72)
    @printf("%-30s", "")
    for l in labels; @printf("%8s", l); end
    println()
    println("-"^70)
    @printf("%-30s", "Standard deviation")
    for k in 1:K; @printf("%8.3f", m_std[k]); end; println()
    @printf("%-30s", "")
    for k in 1:K; @printf(" (%5.3f)", bs_std[k]); end; println()
    @printf("%-30s", "Quarterly autocorrelation")
    for k in 1:K; @printf("%8.3f", m_auto[k]); end; println()
    @printf("%-30s", "")
    for k in 1:K; @printf(" (%5.3f)", bs_auto[k]); end; println()
    println("\nCorrelation matrix:")
    @printf("%-6s", "")
    for l in labels; @printf("%11s", l); end; println()
    for i in 1:K
        @printf("%-6s", labels[i])
        for j in 1:K
            j < i ? @printf("%11s", "—") : @printf("%11.3f", m_corr[i,j])
        end; println()
        @printf("%-6s", "")
        for j in 1:K
            j < i ? @printf("%11s", "") : @printf("  (%7.3f)", bs_corr[i,j])
        end; println()
    end
end

display_table (generic function with 1 method)

In [10]:
#-------------------------------------------------------------------------------
#8. Direct export of results to Latex.
#-------------------------------------------------------------------------------

struct TableOptions
    filename     :: String
    caption      :: String
    label        :: String
    position     :: String
    col_labels   :: Vector{String}
    row_std      :: String
    row_auto     :: String
    corr_title   :: String
    note         :: String
    decimals     :: Int
    se_in_parens :: Bool
end

function TableOptions(spec::ModelSpec;
        filename     :: String         = "$(spec.name).tex",
        caption      :: String         = spec_title(spec),
        label        :: String         = "tab:$(spec.name)",
        position     :: String         = "ht",
        col_labels   :: Vector{String} = ["\$u\$", "\$v\$", "\$v/u\$",
                                          "\$f\$", "\$z\$"],
        row_std      :: String         = "Standard deviation",
        row_auto     :: String         = "Quarterly autocorrelation",
        corr_title   :: String         = "\\textit{Correlation matrix}",
        note         :: String         = "",
        decimals     :: Int            = 3,
        se_in_parens :: Bool           = true)

    TableOptions(filename, caption, label, position, col_labels,
                 row_std, row_auto, corr_title, note, decimals, se_in_parens)
end

function save_latex_table(moments::Tuple,
                          spec   ::ModelSpec,
                          opts   ::TableOptions = TableOptions(spec))

    m_std, bs_std, m_auto, bs_auto, m_corr, bs_corr = moments
    K   = length(opts.col_labels)

    # Printf.Format accepts a runtime string (unlike @sprintf which needs a literal)
    _fmt = Printf.Format("%.$(opts.decimals)f")
    pf(x)     = Printf.format(_fmt, x)
    pfcell(x) = " & " * pf(x)
    pfse(x)   = " & (" * pf(x) * ")"

    open(opts.filename, "w") do io
        println(io, "\\begin{table}[$(opts.position)]")
        println(io, "\\centering")
        println(io, "\\caption{$(opts.caption)}")
        println(io, "\\label{$(opts.label)}")
        println(io, "\\begin{tabular}{l$(repeat('c', K))}")
        println(io, "\\toprule")
        println(io, " & " * join(opts.col_labels, " & ") * " \\\\")
        println(io, "\\midrule")

        # Standard deviations
        row = opts.row_std
        for k in 1:K; row *= pfcell(m_std[k]); end
        println(io, row * " \\\\")
        if opts.se_in_parens
            row = ""
            for k in 1:K; row *= pfse(bs_std[k]); end
            println(io, row * " \\\\")
        end
        println(io, "\\addlinespace")

        # Autocorrelations
        row = opts.row_auto
        for k in 1:K; row *= pfcell(m_auto[k]); end
        println(io, row * " \\\\")
        if opts.se_in_parens
            row = ""
            for k in 1:K; row *= pfse(bs_auto[k]); end
            println(io, row * " \\\\")
        end
        println(io, "\\midrule")

        # Correlation matrix
        println(io, "\\multicolumn{$(K+1)}{l}{$(opts.corr_title)} \\\\")
        println(io, "\\addlinespace")
        for i in 1:K
            row = opts.col_labels[i]
            for j in 1:K
                row *= j < i ? " & " : pfcell(m_corr[i,j])
            end
            println(io, row * " \\\\")
            if opts.se_in_parens
                row = ""
                for j in 1:K
                    row *= j < i ? " & " : pfse(bs_corr[i,j])
                end
                println(io, row * " \\\\")
            end
            i < K && println(io, "\\addlinespace")
        end

        println(io, "\\bottomrule")
        println(io, "\\end{tabular}")

        if !isempty(opts.note)
            println(io, "\\\\[2pt]")
            println(io, "\\begin{minipage}{\\linewidth}")
            println(io, opts.note)
            println(io, "\\end{minipage}")
        end

        println(io, "\\end{table}")
    end

    println("  Saved: $(opts.filename)")
end

save_latex_table(moments::Tuple, spec::ModelSpec; kwargs...) =
    save_latex_table(moments, spec, TableOptions(spec; kwargs...))


save_latex_table (generic function with 2 methods)

In [ ]:

#-------------------------------------------------------------------------------
#9. Main
#-------------------------------------------------------------------------------
#Note: this code runs everything else. 

println("="^60)
println("STEP 1 — Calibrate and discretize")
println("="^60)
par    = calibrate()
N      = 21
y_grid, Π, _ = rouwenhorst(N, par.ρ, par.σ_z)
z_grid = get_z_grid(y_grid, par.b, par.zbar)
z_min  = minimum(z_grid)
@printf("  z_grid ∈ [%.4f, %.4f]\n\n", z_min, maximum(z_grid))

#To add the table of the params we generate ¿¿¿¿¿¿¿¿¿¿¿¿¿¿¿¿¿¿ 
# --- Define all five specifications ---
specs = ModelSpec[
    baseline_spec(par),
    hm_spec(par, z_min),
    hall_spec(par),
    proc_leisure_spec(par),
    proc_gamma_spec(par, 0.1),    # φ=0.1, more reasonable than 1! Revise!!! Make more explicit the choice.
]

println("\n" * "="^60)
println("STEP 2 — Solve equilibrium + simulate each spec (n_sims=10,000)")
println("="^60)

results = Dict{Symbol, Tuple}()

for spec in specs
    @printf("\n>>> %s\n", spec.name)
    θ_grid, c = solve_equilibrium(par, spec, z_grid, Π)
    @printf("    c=%.4f | θ∈[%.4f,%.4f] | θ_mid=%.4f\n",
            c, minimum(θ_grid), maximum(θ_grid), θ_grid[div(N,2)+1])

    std_mat, auto_mat, corr_arr = simulate_spec(par, z_grid, θ_grid, Π)
    results[spec.name] = (std_mat, auto_mat, corr_arr)

    moments = compute_moments(std_mat, auto_mat, corr_arr)
    display_table(spec, moments...)
end

println("\n" * "="^60)
println("STEP 3 — Save LaTeX (one file per spec)")
println("="^60)

for spec in specs
    moments = compute_moments(results[spec.name]...)
    save_latex_table(moments, spec)
end

println("\nDone.")


STEP 1 — Calibrate and discretize
  z_grid ∈ [0.6629, 1.7691]

  [HM] b_hm = 0.6298 (z_min = 0.6629, surplus margin = 0.0331)
  [ProcLeisure] ζ = 0.955 → b(z̄) = 0.9550

STEP 2 — Solve equilibrium + simulate each spec (n_sims=10,000)

>>> baseline
    c=0.2097 | θ∈[0.4269,2.3319] | θ_mid=1.0000

Baseline (Shimer 2005)
                                     u       v     v/u       f       z
----------------------------------------------------------------------
Standard deviation               0.009   0.026   0.034   0.009   0.020
                               (0.002) (0.006) (0.008) (0.002) (0.005)
Quarterly autocorrelation        0.936   0.883   0.912   0.912   0.912
                               (0.020) (0.038) (0.028) (0.028) (0.028)

Correlation matrix:
                u          v        v/u          f          z
u           1.000     -0.937     -0.964     -0.964     -0.964
        (  0.000)  (  0.033)  (  0.025)  (  0.025)  (  0.025)
v               —      1.000      0.996      0.